# 02 — Stage 1 Planner fine-tune

QLoRA fine-tune of `Qwen2.5-7B-Instruct` on the `(english -> pseudocode)`
pairs in `data/stage1_planner/englishtopseudo.jsonl`, after registering the
DSL special tokens (`<PLAN>`, `</PLAN>`, `<STEP>`) from `src/dsl/schema.py`
into the tokenizer — so the model emits them as single tokens instead of
spelling them out character by character.

**This notebook is self-contained: `Runtime -> Run all` on a GPU runtime is
the whole procedure.** Section 2 clones the repo, mounts Drive, and checks
the GPU for you; there are no cells to add by hand. Each section says what
you should see in its output, so you can tell a good run from a bad one as
you go.

Two things worth doing *before* you connect to a runtime, because compute
units are billed on GPU-connected wall time:

1. Edit the section 1 config cell. Colab lets you edit cell source with no
   runtime attached — only *running* needs one.
2. `File -> Save a copy in Drive`, so your edits survive. Opened from the
   GitHub link, this notebook is a read-only render.

`docs/colab_setup.md` in the repo has the rest: choosing a GPU, what
persists between sessions, troubleshooting, and how to reuse the adapter
afterwards.

## 1. Config

Everything you might want to change lives here. Paths are absolute and
derived from `REPO_DIR`, so nothing depends on the runtime's working
directory.

| Setting | Default | Notes |
| --- | --- | --- |
| `REPO_BRANCH` | `main` | Change if the data you want isn't merged yet. |
| `OUTPUT_DIR` | `stage1_planner_qlora` | Runtime-local, so epoch checkpoints die with the session. Point it inside `DRIVE_CHECKPOINT_DIR` if you expect interruptions and want to resume with `trainer.train(resume_from_checkpoint=True)`. |
| `CACHE_MODEL_ON_DRIVE` | `False` | `True` keeps the ~15GB base model on Drive so later sessions skip the download. Only worth it if your Drive has the space to spare. |
| `MAX_SEQ_LEN` | 512 | Prompt + plan, truncated. Raise if you add long plans. |
| `NUM_EPOCHS` | 15 | High on purpose — 50 examples is a proof of concept. |
| `LEARNING_RATE` | 2e-4 | Standard LoRA range (1e-4 to 3e-4). |
| `PER_DEVICE_BATCH_SIZE` × `GRAD_ACCUM_STEPS` | 4 × 4 | Effective batch 16, ~3 steps per epoch. |
| `SPLIT_SEED` | 0 | Seeds the train/eval split. Shared with Stage 2 and the eval notebook — don't change it for one stage only. |
| `EVAL_FRACTION` | 0.2 | 50 examples → 40 train / 10 held out. |
| `EVAL_BATCH_SIZE` | 4 | Eval-only forward passes; no gradients, so it can exceed the train batch size. |

If you hit CUDA out of memory later, drop `PER_DEVICE_BATCH_SIZE` to 2 or 1
and raise `GRAD_ACCUM_STEPS` to keep the effective batch at 16.

In [ ]:
# Where the code and data come from
REPO_URL = "https://github.com/ashfordreyes/pythonllm.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/pythonllm"

DATA_PATH = f"{REPO_DIR}/data/stage1_planner/englishtopseudo.jsonl"
# Stage 2 file, read only to find which rows are safe to exec() -- see
# section 6, where it stratifies the split so the execution-scoring tier
# in 04_eval_pipeline.ipynb isn't left permanently dormant.
STAGE2_DATA = f"{REPO_DIR}/data/stage2_coder/pseudotopython.jsonl"
SCHEMA_PATH = f"{REPO_DIR}/src/dsl/schema.py"

# Where results go. Only Drive survives the runtime shutting down.
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/pythonllm_checkpoints"
OUTPUT_DIR = "stage1_planner_qlora"
CACHE_MODEL_ON_DRIVE = False

# Training
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 512
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

# Held-out split. Both stages split by row index from this seed, so the same
# tasks are held out for Stage 1, Stage 2, and end-to-end eval. Changing the
# seed silently changes which examples the model has already seen.
SPLIT_SEED = 0
EVAL_FRACTION = 0.2
EVAL_BATCH_SIZE = 4

## 2. Setup

Four cells that turn a bare runtime into one this notebook can train on:
check the GPU, install the libraries, mount Drive, clone the repo. Run them
in order — the Drive mount has to happen before anything imports
`transformers` for `CACHE_MODEL_ON_DRIVE` to take effect.

Once these have run clean you don't touch them again for the session.

### 2a. Check the GPU

Colab silently falls back to whatever hardware is free, so confirm you got
what you asked for. Expect ~40GB total memory on an A100 or ~23GB on an L4,
and `True` for both torch checks.

`is_bf16_supported()` must be `True`: section 7 trains with `bf16=True`. A
free T4 reports `False` here — see `docs/colab_setup.md` for why that GPU
isn't practical for this run.

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())  # must be True

### 2b. Install dependencies

`bitsandbytes` provides the 4-bit (NF4) quantization and `peft` provides
LoRA. Takes 1-2 minutes.

If pip reports that it upgraded an already-imported package (usually `torch`
or `transformers`), restart the runtime and re-run from section 1 —
otherwise you get version-mismatch errors several cells later.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl matplotlib

### 2c. Mount Google Drive

**Nothing under `/content` survives the session** — not the clone, not the
model download, not `OUTPUT_DIR`. Drive is the only durable storage Colab
gives you, so this cell mounts it and creates the checkpoint folder that
section 9 copies the trained adapter into.

The first run opens a Google auth popup; approve it. Re-running when already
mounted is harmless.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

if CACHE_MODEL_ON_DRIVE:
    # Must be set before transformers is imported, or it is ignored.
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    print("HF cache on Drive:", os.environ["HF_HOME"])

print("Checkpoints will go to:", DRIVE_CHECKPOINT_DIR)

### 2d. Clone the repo

Opening this notebook does *not* bring the repo with it. The cells below
read the dataset off the runtime's disk and import `SPECIAL_TOKENS` from
`src/dsl/schema.py`, so the files have to be there.

`ashfordreyes/pythonllm` is public, so no token is needed. The cell is safe
to re-run: it pulls instead of cloning if the directory already exists.

Expect `OK: 50 examples`. If the assert fires instead, the clone failed or
`REPO_BRANCH` names a branch that doesn't have the data — fix that before
going on, because the next section is where a missing clone would otherwise
surface as `ModuleNotFoundError: No module named 'dsl'`.

In [ ]:
import pathlib
import sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present; pulling")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

for path in (DATA_PATH, SCHEMA_PATH, STAGE2_DATA):
    assert pathlib.Path(path).exists(), (
        f"missing {path} — the clone failed, or branch '{REPO_BRANCH}' "
        "doesn't have this file"
    )

sys.path.insert(0, f"{REPO_DIR}/src")

n_examples = sum(1 for _ in open(DATA_PATH))
print(f"OK: {n_examples} examples in {DATA_PATH}")

## 3. Tokenizer and DSL special tokens

`SPECIAL_TOKENS` is imported straight from `src/dsl/schema.py` so the
tokenizer always matches the DSL definition in the repo, rather than
hardcoding the token strings here.

Expect `Added 3 new special tokens`. A `0` means either they were already in
the vocab or `SPECIAL_TOKENS` came back empty — worth checking before you
spend a 15GB model download on it, which is why the import and the tokenizer
share a cell.

In [ ]:
from dsl.schema import SPECIAL_TOKENS
from transformers import AutoTokenizer

print("DSL special tokens:", SPECIAL_TOKENS)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": SPECIAL_TOKENS}
)
print(f"Added {num_added} new special tokens; vocab size now {len(tokenizer)}")

## 4. Load the base model in 4-bit

Downloads ~15GB (3-10 minutes; the download is bf16, only the in-memory
weights are 4-bit) and loads it quantized.

The embedding matrix is deliberately *not* resized. `Qwen2.5-7B-Instruct`
ships with `vocab_size: 152064` while its stock tokenizer only reaches id
151664, so the three DSL tokens land at 151665-151667 — already inside the
matrix. Calling `resize_token_embeddings(len(tokenizer))` here would
*shrink* it to 151668 rows, which is what makes the saved adapter carry
multi-GB embedding copies and stop loading onto a stock base model. The
assert below pins that assumption, so swapping in a base model with a
tighter vocab fails loudly here instead of silently.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

# No resize: the DSL token ids must already fit the base embedding matrix.
assert len(tokenizer) <= model.config.vocab_size, (
    f"{BASE_MODEL} has vocab_size={model.config.vocab_size} but the tokenizer "
    f"needs {len(tokenizer)}; this base model would need "
    "resize_token_embeddings() and a different strategy for the DSL rows"
)
print(f"tokenizer needs {len(tokenizer)} of {model.config.vocab_size} rows")

model.config.use_cache = False

## 5. LoRA setup

Rank 16, alpha 32, dropout 0.05, applied to all attention and MLP
projections.

`trainable_token_indices` is what actually teaches the model the DSL
tokens. `prepare_model_for_kbit_training` freezes every base parameter, so
without it the `<PLAN>`, `</PLAN>` and `<STEP>` rows of `embed_tokens` and
`lm_head` would stay at their untrained base-model values no matter how
long you train. Naming the three ids trains exactly those rows — about 21K
extra parameters — instead of the ~1.09B a full `modules_to_save` on both
layers would cost.

Expect roughly 40M trainable parameters out of ~7.6B — well under 1%. A
figure near 1.1B means `trainable_token_indices` was silently ignored;
check the spelling and that `peft` is recent enough to support it.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

new_token_ids = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)
print("DSL token ids:", dict(zip(SPECIAL_TOKENS, new_token_ids)))

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    # Train only the three new rows, on the input and output side both.
    trainable_token_indices={
        "embed_tokens": new_token_ids,
        "lm_head": new_token_ids,
    },
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Load and format the dataset

Each example is formatted with the base model's own chat template so the
planner learns to respond to an instruction-style prompt with a pseudocode
plan, then loss-masked so only the pseudocode completion — not the prompt —
contributes to the loss.

Before any of that, the 50 examples are split 40 train / 10 held out.
`splits.split_dataset` picks the held-out rows *by index* from `SPLIT_SEED`,
and Stage 2 and the eval notebook derive the same indices from the same
seed — so end-to-end eval scores both stages on the same tasks with no split
file to keep in sync.

The split is also stratified: a handful of Stage 2 reference snippets are
safe to `exec()` (no network, GUI, or heavy deps), and
`04_eval_pipeline.ipynb`'s execution-scoring tier needs at least one of them
in the held-out set. Reading `STAGE2_DATA` here just to compute that
priority set -- Stage 1 training only ever uses the pseudocode plans.

Expect `50 examples -> 40 train / 10 eval`.


In [ ]:
from datasets import load_dataset

from eval.scoring import is_executable_example
from splits import EVAL_FRACTION, SPLIT_SEED, read_jsonl, split_dataset

raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# Same executable-reference priority set 04_eval_pipeline.ipynb re-derives,
# so both notebooks resolve the identical stratified split.
stage2_rows = read_jsonl(STAGE2_DATA)
executable_priority = {
    i for i, row in enumerate(stage2_rows) if is_executable_example(row["python_code"])
}

train_raw, eval_raw = split_dataset(raw_dataset, SPLIT_SEED, EVAL_FRACTION, priority=executable_priority)

print(f"{len(raw_dataset)} examples -> {len(train_raw)} train / {len(eval_raw)} eval")
print("\nheld-out task 0:", eval_raw[0]["english"])


In [ ]:
SYSTEM_PROMPT = (
    "You are a planner that turns a task description into a pseudocode plan "
    "using the pythonllm DSL. Respond with only the plan, wrapped in "
    "<PLAN>...</PLAN> and made of <STEP> lines."
)


def format_example(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["english"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = prompt_text + example["pseudocode"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    labels = list(full["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    full["labels"] = labels
    return full


train_dataset = train_raw.map(format_example, remove_columns=train_raw.column_names)
eval_dataset = eval_raw.map(format_example, remove_columns=eval_raw.column_names)

# Sanity check the masking: this should print the plan and nothing else.
row = train_dataset[0]
supervised = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
print(f"{len(supervised)} of {len(row['labels'])} tokens supervised")
print(tokenizer.decode(supervised))

## 7. Train

15 epochs over 40 training examples is ~45 steps — a few minutes on an A100
or L4. Training loss is logged every step and should fall; eval loss on the
10 held-out examples is logged once per epoch.

Watch the gap between them. Training loss falling while eval loss flattens
or rises is memorization, and the fix is fewer epochs — not a better
checkpoint, which is why `load_best_model_at_end` is deliberately off.

If it stays flat or goes `nan`: flat usually means an example got fully
masked (check the supervised-token count printed above is not 0), and `nan`
usually means bf16 isn't really supported — see the section 2a output.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    bf16=True,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    # No load_best_model_at_end on purpose: picking a checkpoint by the loss
    # of 10 examples selects on noise. Read the curve in section 7b instead.
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

## 7b. Loss curve

`trainer.state.log_history` only lives in memory, so dump it before anything
else can end the session. The JSON is the record; the PNG is for reading at a
glance.

Train and eval loss share one axis on purpose — they measure the same thing,
and the gap between them is the whole point. Eval loss climbing while train
loss keeps falling means the run is memorizing 40 examples; that is the signal
to cut `NUM_EPOCHS`.


In [ ]:
import json

from eval.plots import plot_loss_curve

LOSS_HISTORY_PATH = f"{OUTPUT_DIR}/log_history.json"
LOSS_PLOT_PATH = f"{OUTPUT_DIR}/loss_curve.png"

with open(LOSS_HISTORY_PATH, "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

plot_loss_curve(trainer.state.log_history, LOSS_PLOT_PATH)
print(f"Wrote {LOSS_HISTORY_PATH} and {LOSS_PLOT_PATH}")

eval_losses = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
if eval_losses:
    print(f"eval loss: {eval_losses[0]:.4f} (first epoch) -> {eval_losses[-1]:.4f} (last)")

from IPython.display import Image, display
display(Image(LOSS_PLOT_PATH))


## 8. Save the adapter and tokenizer

Both, together: the tokenizer carries the DSL special tokens, and an adapter
loaded against a stock-vocab tokenizer won't work.

Expect `adapter_model.safetensors` to be roughly 160-200MB. Because section
4 left the vocab size alone, PEFT saves a bare LoRA adapter plus the three
trained token rows — not fp32 copies of the whole `embed_tokens` and
`lm_head` matrices, which would push the file past 4GB.

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved LoRA adapter + tokenizer to {FINAL_DIR}")

## 9. Copy the adapter to Google Drive

**Do not skip this.** `FINAL_DIR` is on the runtime's disk, which is
destroyed when the session ends, disconnects, or is reclaimed for idling.
This is the one artifact of the whole run you can't cheaply reproduce.

The loss history and its PNG go along with it: a checkpoint with no record
of how it trained is much harder to compare against the next one.

Drive is already mounted from section 2c.

In [ ]:
DRIVE_ADAPTER_DIR = f"{DRIVE_CHECKPOINT_DIR}/stage1_planner"

!rm -rf {DRIVE_ADAPTER_DIR}
!cp -r {FINAL_DIR} {DRIVE_ADAPTER_DIR}
# The loss curve and its raw log travel with the adapter -- without them
# there's no record of how this checkpoint trained.
!cp {LOSS_HISTORY_PATH} {LOSS_PLOT_PATH} {DRIVE_ADAPTER_DIR}/
!ls {DRIVE_ADAPTER_DIR}
print(f"Adapter, loss history and loss curve saved to {DRIVE_ADAPTER_DIR}")

## 10. Quick sanity check

Runs one **held-out** example through the fine-tuned model — a task the
model never trained on, so the output is a (very small) quality signal
rather than a recall test. The reference plan is printed above it to compare
against. It decodes with
`skip_special_tokens=False`, so you should literally see `<PLAN>`, `<STEP>`
and `</PLAN>` in the output.

If the model spells out `<`, `PLAN`, `>` as separate pieces, the tokenizer
in use is missing the special tokens — load the one saved next to the
adapter, not a fresh one from the hub.

In [ ]:
model.eval()
test_english = eval_raw[0]["english"]  # held out: the model never saw this
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_english},
]
# return_dict=True pinned explicitly: without it, whether apply_chat_template
# returns a raw tensor or a BatchEncoding depends on the transformers
# version, and a BatchEncoding has no .shape (see docs/colab_setup.md).
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)
inputs = encoded["input_ids"]

with torch.no_grad():
    output_ids = model.generate(
        inputs,
        attention_mask=encoded["attention_mask"],
        max_new_tokens=MAX_SEQ_LEN,
        do_sample=False,
    )

print("REFERENCE:", eval_raw[0]["pseudocode"])
print("\nINPUT:", test_english)
print("OUTPUT:", tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=False))

## 11. Release the runtime

Compute units are billed on GPU-connected wall time, and closing the browser
tab does **not** disconnect — Colab keeps the runtime alive in the
background, still billing. This cell ends the session and frees the GPU.

It kills the kernel, so only run it once section 9 has confirmed the adapter
is on Drive. Comment it out if you still want to poke at the in-memory
model.

In [ ]:
from google.colab import runtime

runtime.unassign()